In [138]:
import openseespywin as ops
import opsvis as opsv 
import numpy as np
import ipywidgets as widgets
import os
import matplotlib.pyplot as plt
import math
import opstool as opst
import eurocodepy as ecpy
import time as tt
# import openseespy.postprocessing.Get_Rendering as opsplt

In [139]:
import openseespy.opensees as ops

#Soil Element Properties
elasto_mat_tag = 1
thickness = 1.0
type = "PlaneStrain"

# Soil Material properties
E = 200e6         # Elastic modulus in Pa
nu = 0.3          # Poisson's ratio
rho = 0.0         # Density

# Storage for created nodes and elements
created_nodes = []
created_elements = []
block_registry = {}

def generate_block(start_x, start_y, length, height, node_offset, elem_offset, mat_tag, box_width, box_height, block_name="block"):
    num_x = int(length / box_width)
    num_y = int(height / box_height)

    def block_node_id(i, j):
        return node_offset + j * (num_x + 1) + i + 1

    block_nodes = []
    block_elements = []

    # Create nodes
    for j in range(num_y + 1):
        for i in range(num_x + 1):
            nid = block_node_id(i, j)
            x = start_x + i * box_width
            y = start_y + j * box_height
            ops.node(nid, x, y)
            created_nodes.append((nid, x, y))
            block_nodes.append(nid)

    # Create elements
    eid = elem_offset
    for j in range(num_y):
        for i in range(num_x):
            n1 = block_node_id(i, j)
            n2 = block_node_id(i + 1, j)
            n3 = block_node_id(i + 1, j + 1)
            n4 = block_node_id(i, j + 1)
            # element quadUP $eleTag $iNode $jNode $kNode $lNode $thick $matTag $bulk $fmass $hPerm $vPerm <$b1=0 $b2=0 $t=0>
            ops.element("quad", eid, n1, n2, n3, n4, thickness, type, mat_tag)
            created_elements.append((eid, n1, n2, n3, n4))
            block_elements.append(eid)
            eid += 1

    # Register block
    block_registry[block_name] = {
        "nodes": block_nodes,
        "elements": block_elements,
    }

    return (node_offset + (num_x + 1) * (num_y + 1), elem_offset + num_x * num_y)

# Start model
ops.wipe()
ops.model("Basic", "-ndm", 2, "-ndf", 2)

ops.nDMaterial(
            "PressureDependMultiYield02",
            5,
            2,
            1.8,
            9600.0,
            27000.0,
            36,
            0.1,
            101.0,
            0.0,
            26,
            0.067,
            0.23,
            0.06,
            0.27,
            20,
            5.0,
            3.0,
            1.0,
            0.0,
            0.77,
            0.9,
            0.02,
            0.7,
            101.0,
        )
        # element thickness
        # body force in x-direction
        # body force in y-direction
        # create wrapper material for initial state analysis
elasto_mat_tag = 1
ops.nDMaterial("InitialStateAnalysisWrapper", 1, 5, 2)

    #elasto_mat_tag = 100
    # Define material
    #ops.nDMaterial("ElasticIsotropic", elasto_mat_tag, E, nu, rho)

    # Define contact material
contact_mat = 2
ops.nDMaterial("ContactMaterial2D", contact_mat, 0.1, 1000.0, 0.0, 0.0)

# Generate one block starting from node 1 and element 1
node_offset = 0
elem_offset = 1
node_offset, elem_offset = generate_block(45, 16, 8.0, 2.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="OAE_Soil_block")
node_offset, elem_offset = generate_block(45, 19, 1.5, 10.0, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="Fill_Soil_block")
node_offset, elem_offset = generate_block(45, 29.5, 3.0, 2.0, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="F1_Soil_block")
node_offset, elem_offset = generate_block(45, 32, 8.0, 8.0, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="Fill_Soil_block")
#node_offset, elem_offset = generate_block(8.5, 0.25, 3.0, 7.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="F1_Soil_block")


In [140]:
# Fixity definitions
fixXY = [1, 1]
fixXonly = [1, 0]
fixYonly = [0, 1]

# Inputs
start_id = 1
end_id = 17
step = 1

# Generate node list
bottom_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in bottom_nodes:
    ops.fix(nid, *fixXY)


In [141]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

511


In [142]:
# Inputs
start_id = 18
end_id = 86
step = 17

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [143]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)


Last element ID: 420


In [144]:
# Add paired quads using incremental node range
bottom_start = 86
bottom_end = 89
top_start = 103
top_end = 106
step = 1  # step can be changed

bottom_nodes = list(range(bottom_start, bottom_end + 1, step))
top_nodes = list(range(top_start, top_end + 1, step))

start_elem_id = last_eid + 1
paired_quad_ids = []

for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

In [145]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

Last element ID: 423


In [146]:
# Add paired quads using incremental node range
bottom_start = 183
bottom_end = 186
top_start = 187
top_end = 190
step = 1  # step can be changed

bottom_nodes = list(range(bottom_start, bottom_end + 1, step))
top_nodes = list(range(top_start, top_end + 1, step))

start_elem_id = last_eid + 1
paired_quad_ids = []

for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

In [147]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Add paired quads using incremental node range
bottom_start = 215
bottom_end = 221
top_start = 222
top_end = 228
step = 1  # step can be changed

bottom_nodes = list(range(bottom_start, bottom_end + 1, step))
top_nodes = list(range(top_start, top_end + 1, step))

start_elem_id = last_eid + 1
paired_quad_ids = []

for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 426


In [148]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

511


In [149]:
stitch_node = [
[51.500 , 18.565],
[49.500 , 19.000],
[49.000 , 19.000],
[48.500 , 19.000],
[48.000 , 19.000],
[47.500 , 19.000],
[47.000 , 19.000],
[47.000 , 19.500],
[47.000 , 20.000],
[47.000 , 20.500],
[47.000 , 21.000],
[47.000 , 21.500],
[47.000 , 22.000],
[46.757 , 22.500],
[46.618 , 22.985],
[46.560 , 23.500],
[46.613 , 26.453],
[47.000 , 28.000],
[47.000 , 28.500],
[47.000 , 29.000],
[47.500 , 29.000],
[48.500 , 30.000],
[48.500 , 30.500],
[48.500 , 31.000],
[48.500 , 31.500],
[49.000 , 31.500],
[49.500 , 31.500],
[50.000 , 31.500],
[50.500 , 31.500],
[51.000 , 31.500],
[51.500 , 31.500],
[49.000 , 31.000],
[49.500 , 31.000],
[50.000 , 31.000],
[49.514 , 30.778],
[49.158 , 30.778],
[49.321 , 30.710],
[49.000 , 30.500],
[47.305 , 28.654],
[47.500 , 19.500],
[47.500 , 20.000],
[47.500 , 20.500],
[47.500 , 21.000],
[48.000 , 19.500],
[48.500 , 19.500],
[49.000 , 19.500],
[48.000 , 20.000],
[48.500 , 20.000],
[48.000 , 20.500],
[47.787 , 20.798],
[47.611 , 21.024],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(stitch_node):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["sheet_pile_nodes"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [150]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

562


In [151]:
slave_node_A = [
[52.500 , 31.500],
[52.000 , 31.500],
[51.673 , 31.414],
[51.179 , 31.292],
[50.713 , 31.137],
[50.203 , 30.923],
[49.762 , 30.694],
[49.321 , 30.419],
[48.910 , 30.116],
[48.537 , 29.794],
[48.172 , 29.426],
[47.850 , 29.047],
[47.542 , 28.622],
[47.261 , 28.157],
[47.034 , 27.704],
[46.833 , 27.207],
[46.674 , 26.698],
]


start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(slave_node_A):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["slave_nodes_A"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [152]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

579


In [153]:
slave_node_B = [
[46.611 , 23.552],
[46.741 , 23.068],
[46.923 , 22.555],
[47.126 , 22.102],
[47.379 , 21.636],
[47.664 , 21.200],
[47.959 , 20.817],
[48.314 , 20.423],
[48.700 , 20.058],
[49.109 , 19.730],
[49.547 , 19.433],
[49.959 , 19.198],
[50.398 , 18.988],
[50.917 , 18.790],
[51.391 , 18.650],
]


start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(slave_node_B):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["slave_nodes_B"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [154]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [89, 90, 91, 92, 93, 94, 95]
top_nodes    = [106, 517, 516, 515, 514, 513, 512]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}


Last element ID: 432


In [155]:
ops.element("quad", 439, 95, 590, 589, 512, thickness, type, elasto_mat_tag)
ops.element("quad", 440, 95, 96, 591, 590, thickness, type, elasto_mat_tag)
ops.element("quad", 441, 96, 97, 592, 591, thickness, type, elasto_mat_tag)
ops.element("quad", 442, 97, 98, 593, 592, thickness, type, elasto_mat_tag)
ops.element("quad", 443, 98, 99, 511, 593, thickness, type, elasto_mat_tag)

In [156]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [106, 517, 516, 515, 514, 513, 512]
top_nodes    = [110, 518, 550, 554, 555, 556, 589]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 443


In [157]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [110, 518, 550, 554, 555, 588]
top_nodes    = [114, 519, 551, 557, 558,587]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 449


In [158]:
ops.element("quad", 455, 555, 556, 589, 588, thickness, type, elasto_mat_tag)
ops.element("quad", 456, 557, 558, 587, 586, thickness, type, elasto_mat_tag)


In [159]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [114, 519, 551, 557,]
top_nodes    = [118, 520, 552, 559,]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 456


In [160]:
ops.element("quad", 460, 557, 586, 585, 559, thickness, type, elasto_mat_tag)
ops.element("quad", 461, 552, 559, 585, 560, thickness, type, elasto_mat_tag)
ops.element("quad", 462, 552, 560, 561, 553, thickness, type, elasto_mat_tag)
ops.element("quad", 463, 560, 585, 584, 561, thickness, type, elasto_mat_tag)

In [161]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [118, 520, 552,]
top_nodes    = [122, 521, 553,]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 463


In [162]:
ops.element("quad", 466, 521, 553, 561, 584, thickness, type, elasto_mat_tag)
ops.element("quad", 467, 122, 521, 522, 126, thickness, type, elasto_mat_tag)
ops.element("quad", 468, 521, 584, 583, 522, thickness, type, elasto_mat_tag)
ops.element("quad", 469, 126, 522, 523, 130, thickness, type, elasto_mat_tag)
ops.element("quad", 470, 522, 583, 582, 523, thickness, type, elasto_mat_tag)
ops.element("quad", 471, 130, 523, 524, 134, thickness, type, elasto_mat_tag)
ops.element("quad", 472, 523, 582, 581, 524, thickness, type, elasto_mat_tag)
ops.element("quad", 473, 134, 524, 525, 138, thickness, type, elasto_mat_tag)
ops.element("quad", 474, 524, 581, 580, 525, thickness, type, elasto_mat_tag)
ops.element("quad", 475, 138, 525, 526, 142, thickness, type, elasto_mat_tag)
ops.element("quad", 476, 525, 580, 579, 526, thickness, type, elasto_mat_tag)
ops.element("quad", 477, 142, 526, 579, 146, thickness, type, elasto_mat_tag)
ops.element("quad", 478, 162, 527, 578, 166, thickness, type, elasto_mat_tag)
ops.element("quad", 479, 166, 578, 577, 170, thickness, type, elasto_mat_tag)
ops.element("quad", 480, 170, 577, 576, 174, thickness, type, elasto_mat_tag)
ops.element("quad", 481, 174, 576, 528, 178, thickness, type, elasto_mat_tag)
ops.element("quad", 482, 178, 528, 529, 182, thickness, type, elasto_mat_tag)
ops.element("quad", 483, 576, 575, 529, 528, thickness, type, elasto_mat_tag)
ops.element("quad", 484, 575, 574, 549, 529, thickness, type, elasto_mat_tag)

In [163]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [182, 529, 549, 574,]
top_nodes    = [186, 530, 531, 573,]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 484


In [164]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [186, 530, 531, 573,]
top_nodes    = [190, 191, 192, 193,]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 487


In [165]:
ops.element("quad", 491, 573, 572, 200, 193, thickness, type, elasto_mat_tag)
ops.element("quad", 492, 572, 571, 532, 200, thickness, type, elasto_mat_tag)
ops.element("quad", 493, 200, 532, 533, 207, thickness, type, elasto_mat_tag)
ops.element("quad", 494, 571, 570, 533, 532, thickness, type, elasto_mat_tag)
ops.element("quad", 495, 570, 569, 548, 533, thickness, type, elasto_mat_tag)

In [166]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [207, 533, 548, ]
top_nodes    = [214, 534, 542, ]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 495


In [167]:
ops.element("quad", 498, 548, 569, 547, 546, thickness, type, elasto_mat_tag)
ops.element("quad", 499, 569, 568, 545, 547, thickness, type, elasto_mat_tag)
ops.element("quad", 500, 547, 545, 543, 546, thickness, type, elasto_mat_tag)
ops.element("quad", 501, 548, 546, 543, 542, thickness, type, elasto_mat_tag)
ops.element("quad", 502, 568, 544, 543, 545, thickness, type, elasto_mat_tag)
ops.element("quad", 503, 568, 567, 538, 544, thickness, type, elasto_mat_tag)

In [168]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [214, 534, 542, 543, 544 ]
top_nodes    = [221, 535, 536, 537, 538]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 503


In [169]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [221, 535, 536, 537, 538, 539, 540, 541]
top_nodes    = [228, 229, 230, 231, 232, 233, 234, 235]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 507


In [170]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [567, 566, 565, 564]
top_nodes    = [538, 539 ,540, 541]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 514


In [171]:
ops.node(594, 53, 31.5)
ops.node(595, 51.824, 31.4537)

In [172]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

# Example paired node lists
bottom_nodes = [541, 563, 562, 594]
top_nodes    = [235, 236 ,237, 238]

# Element parameters
start_elem_id = last_eid + 1
paired_quad_ids = []

# Create quad elements from explicit node pairs
for i in range(len(bottom_nodes) - 1):
    n1 = bottom_nodes[i]
    n2 = bottom_nodes[i + 1]
    n3 = top_nodes[i + 1]
    n4 = top_nodes[i]
    eid = start_elem_id + i
    ops.element("quad", eid, n1, n2, n3, n4, thickness, type, elasto_mat_tag)
    created_elements.append((eid, n1, n2, n3, n4))
    paired_quad_ids.append(eid)

# Register
block_registry["paired_quads"] = {
    "nodes": bottom_nodes + top_nodes,
    "elements": paired_quad_ids
}

Last element ID: 517


In [173]:
ops.element("quad", 521, 564, 595, 563, 541, thickness, type, elasto_mat_tag)

In [174]:
# Inputs
start_id = 103
end_id = 187
step = 4

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [175]:
# Inputs
start_id = 194
end_id = 222
step = 7

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [176]:
# Inputs
start_id = 239
end_id = 494
step = 17

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [177]:
# Inputs
start_id = 34
end_id = 102
step = 17

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [178]:
# Inputs
start_id = 238
end_id = 510
step = 17

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [179]:
ops.fix(594, *fixYonly)

In [180]:
opst.vis.plotly.plot_model(show_node_numbering=True)

In [181]:
ops.model("Basic", "-ndm", 2, "-ndf", 3)

In [182]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

596


In [183]:
sheet_pile_node = [
[53.000 , 31.300],
[52.500 , 31.280],
[52.004 , 31.220],
[51.514 , 31.122],
[51.033 , 30.985],
[50.565 , 30.810],
[50.112 , 30.599],
[49.677 , 30.352],
[49.263 , 30.072],
[48.873 , 29.760],
[48.508 , 29.417],
[48.172 , 29.047],
[47.866 , 28.652],
[47.593 , 28.233],
[47.354 , 27.795],
[47.150 , 27.338],
[46.983 , 26.867],
[46.854 , 26.384],
[46.763 , 25.893],
[46.712 , 25.395],
[46.700 , 24.896],
[46.728 , 24.396],
[46.796 , 23.901],
[46.903 , 23.413],
[47.048 , 22.934],
[47.230 , 22.469],
[47.449 , 22.020],
[47.703 , 21.589],
[47.990 , 21.180],
[48.308 , 20.795],
[48.656 , 20.436],
[49.032 , 20.106],
[49.432 , 19.807],
[49.855 , 19.540],
[50.298 , 19.308],
[50.758 , 19.112],
[51.232 , 18.953],
[51.717 , 18.832],
[52.210 , 18.749],
[52.708 , 18.706],
[53.000 , 18.700],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(sheet_pile_node):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["sheet_pile_nodes"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [184]:
ops.fix(636, 0, 1, 0)

In [185]:
master_node_ids = block_registry["sheet_pile_nodes"]["nodes"]
print(master_node_ids)

[596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636]


In [186]:
# Inputs
lagrange_start_id = 700
num_lagrange_nodes = 40
x_coord = 53.0
y_coord = 18.7

# Output list
lagrange_node_ids = []

# Create nodes
for i in range(num_lagrange_nodes):
    nid = lagrange_start_id + i
    ops.node(nid, x_coord, y_coord)
    created_nodes.append((nid, x_coord, y_coord))
    lagrange_node_ids.append(nid)

# Register block
block_registry["lagrange_nodes"] = {
    "nodes": lagrange_node_ids,
    "elements": []
}

In [187]:
lagrange_node_ids = block_registry["lagrange_nodes"]["nodes"]
print(lagrange_node_ids)

[700, 701, 702, 703, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739]


In [188]:
slaveNode_A = block_registry["slave_nodes_A"]["nodes"]
slaveNode_B = block_registry["slave_nodes_B"]["nodes"]
print(slaveNode_A)
print(slaveNode_B)

[562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578]
[579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593]


In [189]:
final_slave_nodes = slaveNode_A + [162, 158, 154, 150, 146] + slaveNode_B + [100, 101, 102]


In [190]:
master_nodes = master_node_ids
slave_nodes = final_slave_nodes
lagrange_nodes = lagrange_node_ids

contact_elem_ids = []
beam_start_id = 1000
for i in range(len(slave_nodes)):
    tag = beam_start_id + i
    iN = master_nodes[i]
    jN = master_nodes[i + 1]
    sN = slave_nodes[i]
    IN = lagrange_nodes[i]

    ops.element("BeamContact2D", tag, iN, jN, sN, IN, contact_mat, 0.5, 1e-10, 1e-10)
    created_elements.append((tag, iN, jN, sN, IN))
    contact_elem_ids.append(tag)

block_registry["contact_beams_right"] = {
    "nodes": master_nodes + slave_nodes + lagrange_nodes,
    "elements": contact_elem_ids
}

In [191]:
# Inputs
beam_nodes = master_node_ids  # Example node list (ordered start → end)
transFTag = 1
beam_secTag = 1
intTag = 401
Nint = 3
beam_start_id = 3000  # Starting element tag


# Geometry transformation and section definition
ops.geomTransf("Linear", transFTag)
ops.section("Elastic", beam_secTag, 200e6, 0.5, 0.000975)
ops.beamIntegration("Legendre", intTag, beam_secTag, Nint)

# Create elements
beam_elem_ids = []
for i in range(len(beam_nodes) - 1):
    sN = beam_nodes[i]
    eN = beam_nodes[i + 1]
    eid = beam_start_id + i
    ops.element("dispBeamColumn", eid, sN, eN, transFTag, intTag)
    created_elements.append((eid, sN, eN))
    beam_elem_ids.append(eid)

# Register block
block_registry["dispBeamColumn"] = {
    "nodes": beam_nodes,
    "elements": beam_elem_ids
}

In [192]:
beams_cols_ele = block_registry["dispBeamColumn"]["elements"]
print(beams_cols_ele)

[3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3030, 3031, 3032, 3033, 3034, 3035, 3036, 3037, 3038, 3039]


In [193]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

740


In [194]:
pad_foundation = [
[45.250 , 40.250],
[45.750 , 40.250],
[46.250 , 40.250],
[46.750 , 40.250],
[47.250 , 40.250],
[47.750 , 40.250],
[48.250 , 40.250],
[48.750 , 40.250],
[49.250 , 40.250],
[49.750 , 40.250],
[50.250 , 40.250],
[50.750 , 40.250],
[51.250 , 40.250],
[51.750 , 40.250],
[52.250 , 40.250],
[52.750 , 40.250],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(pad_foundation):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["pad_foundation"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [195]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

756


In [196]:
pad_founda_nodes = block_registry["pad_foundation"]["nodes"]
print(pad_founda_nodes)

[740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755]


In [197]:
# Inputs
beam_nodes = pad_founda_nodes  # Example node list (ordered start → end)
beam_start_id = 800  # Starting element tag

# Create elements
beam_elem_ids = []
for i in range(len(beam_nodes) - 1):
    sN = beam_nodes[i]
    eN = beam_nodes[i + 1]
    eid = beam_start_id + i
    ops.element("dispBeamColumn", eid, sN, eN, transFTag, intTag)
    created_elements.append((eid, sN, eN))
    beam_elem_ids.append(eid)

# Register block
block_registry["pad_footing_beam"] = {
    "nodes": beam_nodes,
    "elements": beam_elem_ids
}

In [198]:
ground_slave_nodes = [495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509] # same as beam qty

In [199]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

756


In [200]:
# Inputs
lagrange_start_id = 800
num_lagrange_nodes = 15 # same as beam qty
x_coord = 45.25
y_coord = 40.25

# Output list
lagrange_node_ids = []

# Create nodes
for i in range(num_lagrange_nodes):
    nid = lagrange_start_id + i
    ops.node(nid, x_coord, y_coord)
    created_nodes.append((nid, x_coord, y_coord))
    lagrange_node_ids.append(nid)

# Register block
block_registry["lagrange_nodes_set2"] = {
    "nodes": lagrange_node_ids,
    "elements": []
}

In [201]:
lagrange_node_ids_set_b = block_registry["lagrange_nodes_set2"]["nodes"]
print(lagrange_node_ids_set_b)

[800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814]


In [202]:
last_eid = max(ops.getEleTags()) if ops.getEleTags() else 0
print("Last element ID:", last_eid)

Last element ID: 3039


In [203]:
master_nodes = pad_founda_nodes
slave_nodes = ground_slave_nodes
lagrange_nodes = lagrange_node_ids_set_b

contact_elem_ids = []
beam_start_id = 3050
for i in range(len(slave_nodes)):
    tag = beam_start_id + i
    iN = master_nodes[i]
    jN = master_nodes[i + 1]
    sN = slave_nodes[i]
    IN = lagrange_nodes[i]

    ops.element("BeamContact2D", tag, iN, jN, sN, IN, contact_mat, 0.5, 1e-10, 1e-10)
    created_elements.append((tag, iN, jN, sN, IN))
    contact_elem_ids.append(tag)

block_registry["contact_beams_right"] = {
    "nodes": master_nodes + slave_nodes + lagrange_nodes,
    "elements": contact_elem_ids
}

In [204]:
opst.vis.plotly.plot_model(show_node_numbering=True)

In [205]:
ops.fix(740, 1, 0, 0)
ops.fix(755, 1, 0, 0)

In [206]:
ODB = opst.post.CreateODB(odb_tag=1, model_update=True)